# TP 5 — Jointures, audit référentiel et entonnoir de conversion

**Big Data Engineering — Master 1 — DMI / FST / UCAD**

Séance 5 — Spark SQL avancé et analyse métier.

**Consignes :**
1. Complétez toutes les cellules marquées `# === À COMPLÉTER ===` (les `...` indiquent les trous) ;
2. Remplissez le **tableau de relevés** au fil des exercices ;
3. Rédigez les réponses aux questions de réflexion (cellules « *Votre réponse :* ») ;
4. Exécutez le notebook **de bout en bout** (« Restart & Run All ») avant de le pousser **avec ses sorties**.

**Livrable :** ce notebook, dans `notebooks/` de votre dépôt GitHub, poussé **avant la séance 6**.


## 0. Vérification de l'environnement

Comme aux TP précédents : Python ≥ 3.9, PySpark installé, données du fil rouge
générées à l'**échelle 0.1** avec la **graine 42** (par défaut du script).
Si le dossier `data/` est absent (ou après un reset Colab), dé-commentez la
cellule de génération.

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0], "-", platform.system())

import pyspark
print("PySpark :", pyspark.__version__)

In [ ]:
# Si necessaire (environ 1 minute a l'echelle 0.1) :
# !python3 generate_data.py --scale 0.1 --outdir ./data

import os
attendus = ["customers.csv", "products.csv", "orders.csv",
            "order_items.csv", "payments.json", "events.json"]
manquants = [f for f in attendus if not os.path.exists(os.path.join("data", f))]
print("Fichiers manquants :", manquants if manquants else "aucun - OK")

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("TP5_jointures")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "- session prete")

## Tableau de relevés

Remplissez ce dictionnaire **au fil du TP** (ré-exécutez la cellule après
chaque mise à jour). Graine 42 oblige : vos valeurs doivent être identiques
à celles de vos voisins — comparez-les, c'est un contrôle gratuit.

In [ ]:
releves = {
    "R1_count_commandes_clean":      None,  # partie A
    "R2_count_apres_jointure":       None,  # partie B1
    "R3_ca_sentinelle_avant_apres":  None,  # partie B3 (tuple)
    "R4_orphelines_count_et_part":   None,  # partie C1 (tuple)
    "R5_ca_orphelines":              None,  # partie C2
    "R6_entonnoir_sessions":         None,  # partie D1 (tuple de 4)
    "R7_taux_etape_a_etape":         None,  # partie D1 (tuple de 3, en %)
    "R8_purchase_orphelins":         None,  # partie D3
}
releves

## Partie A — Mise en place et reconstruction de `commandes_clean` (20 min)

On recharge les quatre tables « transactionnelles » et on reconstruit la vue
nettoyée du TP 4. Rappel du piège : ~1 % des montants de `orders.csv` portent
un suffixe « FCFA », ce qui force **toute la colonne** en `string`.

In [ ]:
orders = spark.read.option("header", True).csv("data/orders.csv")
customers = spark.read.option("header", True).csv("data/customers.csv")
products = spark.read.option("header", True).csv("data/products.csv")
items = spark.read.option("header", True).csv("data/order_items.csv")

orders.createOrReplaceTempView("commandes_brutes")
customers.createOrReplaceTempView("clients")
products.createOrReplaceTempView("produits")
items.createOrReplaceTempView("lignes_commande")

print("orders        :", orders.count())
print("customers     :", customers.count())
print("products      :", products.count())
print("order_items   :", items.count())

### A1 — La vue `commandes_clean`

Recréez la vue du TP 4 : montant nettoyé (suppression de tout caractère non
numérique) puis casté en `BIGINT`.

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW commandes_clean AS
SELECT order_id, customer_id, date_commande, statut, canal,
       CAST(regexp_replace(..., '[^0-9]', '') AS ...) AS montant_total_fcfa
FROM commandes_brutes
""")

total_commandes = spark.table("commandes_clean").count()
releves["R1_count_commandes_clean"] = total_commandes
print("Releve R1 :", total_commandes)

## Partie B — Jointures multi-tables contrôlées (35 min)

### B1 — Le CA par région, avec la discipline du cours

**Comptage avant, jointure, comptage après** — puis une phrase d'explication.

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventes_regions AS
SELECT o.*, c.ville, c.region
FROM   commandes_clean o
...    clients c ON o.customer_id = c.customer_id
""")

apres = spark.table("ventes_regions").count()
releves["R2_count_apres_jointure"] = apres
print("Avant :", total_commandes, "| Apres :", apres)

**Question B1 :** le comptage après jointure est-il égal au relevé R1 ?
Expliquez la différence en une ou deux phrases (vous vérifierez votre
hypothèse en partie C).

*Votre réponse :* …

### B1 (suite) — Le CA par région

Sur les commandes **livrées** uniquement, trié décroissant.

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
SELECT region,
       SUM(...) AS ca_fcfa,
       COUNT(*)  AS nb_commandes
FROM   ventes_regions
WHERE  statut = '...'
GROUP BY region
ORDER BY ca_fcfa DESC
""").show(20, truncate=False)

### B2 — Le top produits : quatre tables

Chaîne complète `lignes_commande` → `commandes_clean` → `clients` → `produits`.

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
SELECT p.categorie, p.nom_produit,
       SUM(i.quantite * i.prix_unitaire_fcfa) AS ca
FROM   lignes_commande i
JOIN   commandes_clean o ON i.order_id    = o.order_id
JOIN   clients c         ON ... = ...
JOIN   produits p        ON ... = ...
WHERE  o.statut = 'livree'
GROUP BY p.categorie, p.nom_produit
ORDER BY ca DESC
LIMIT 10
""").show(truncate=False)

### B3 — L'agrégat sentinelle

Le CA total (livrées) **avant** et **après** la jointure d'enrichissement.
S'ils diffèrent, une jointure a perdu ou dupliqué des lignes.

In [ ]:
# === À COMPLÉTER ===
ca_avant = spark.sql(
    "SELECT SUM(montant_total_fcfa) FROM commandes_clean "
    "WHERE statut = 'livree'").first()[0]
ca_apres = spark.sql(
    "SELECT ... FROM ventes_regions WHERE ...").first()[0]

releves["R3_ca_sentinelle_avant_apres"] = (ca_avant, ca_apres)
print("CA avant :", ca_avant)
print("CA apres :", ca_apres)
print("Ecart    :", ca_avant - ca_apres)

**Question B3 :** l'écart observé est-il une **perte** ou une
**duplication** ? Quel indice vous permet de trancher sans même regarder
la partie C ?

*Votre réponse :* …

## Partie C — L'enquête : commandes orphelines (30 min)

### C1 — Compter et vérifier la partition

La question d'audit du cours : *toutes les commandes ont-elles un client
connu ?*

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW commandes_orphelines AS
SELECT o.*
FROM   commandes_clean o
LEFT ... JOIN clients c ON o.customer_id = c.customer_id
""")

nb_orphelines = spark.table("commandes_orphelines").count()
part = 100.0 * nb_orphelines / total_commandes
releves["R4_orphelines_count_et_part"] = (nb_orphelines, round(part, 3))
print(f"Orphelines : {nb_orphelines} ({part:.3f} % du total)")

Le **test de partition** du cours — obligatoire avant d'aller plus loin :
`count(semi) + count(anti) = count(gauche)`.

In [ ]:
# === À COMPLÉTER ===
nb_valides = spark.sql("""
SELECT COUNT(*) FROM commandes_clean o
LEFT ... JOIN clients c ON o.customer_id = c.customer_id
""").first()[0]

assert nb_valides + nb_orphelines == total_commandes, "partition cassee !"
print("Partition exacte verifiee :",
      nb_valides, "+", nb_orphelines, "=", total_commandes)

### C2 — Caractériser avant de décider

Échantillon, concentration temporelle, enjeu financier — la démarche
d'enquête du cours.

In [ ]:
# === À COMPLÉTER ===
# 1. Echantillon : format des customer_id en cause ?
spark.table("commandes_orphelines").show(10, truncate=False)

# 2. Concentration temporelle ?
spark.sql("""
SELECT date_format(date_commande, 'yyyy-MM') AS mois, COUNT(*) AS n
FROM   commandes_orphelines
GROUP BY 1 ORDER BY 1
""").show(30)

# 3. Enjeu financier ?
ca_orph = spark.sql(
    "SELECT SUM(...) FROM commandes_orphelines").first()[0]
releves["R5_ca_orphelines"] = ca_orph
print("CA porte par les orphelines :", ca_orph, "FCFA")

**Question C2 — la décision.** En vous appuyant sur vos relevés R4 et R5,
rédigez en 3–4 lignes la décision que vous prenez pour la suite des analyses
(quarantaine ? exclusion documentée ? conservation en « région inconnue » ?)
et sa justification.

*Votre réponse :* …

**Question piège :** réécrivez l'anti-jointure avec `NOT IN`. Obtenez-vous
le même compte ? Dans quel cas ces deux écritures divergeraient-elles ?

In [ ]:
# === À COMPLÉTER ===
nb_not_in = spark.sql("""
SELECT COUNT(*) FROM commandes_clean
WHERE customer_id NOT IN (SELECT customer_id FROM clients)
""").first()[0]
print("NOT IN :", nb_not_in, "| LEFT ANTI :", nb_orphelines)

## Partie D — L'entonnoir de conversion (40 min)

### D1 — Les comptages par étape

On charge `events.json` (~330 000 événements à l'échelle 0.1) et on compte
des **sessions distinctes** par étape — jamais des événements bruts.

In [ ]:
events = spark.read.json("data/events.json")
events.createOrReplaceTempView("events")
print("Evenements :", events.count())
events.printSchema()

In [ ]:
# === À COMPLÉTER ===
funnel = spark.sql("""
SELECT
  COUNT(DISTINCT session_id)                        AS sessions,
  COUNT(DISTINCT CASE WHEN event_type = 'view_product'
                      THEN session_id END)          AS vues,
  COUNT(DISTINCT CASE WHEN event_type = '...'
                      THEN session_id END)          AS paniers,
  COUNT(DISTINCT CASE WHEN event_type = '...'
                      THEN session_id END)          AS achats
FROM events
""").first()

s1, s2, s3, s4 = funnel
releves["R6_entonnoir_sessions"] = (s1, s2, s3, s4)
print("Sessions :", s1, "| Vues :", s2, "| Paniers :", s3, "| Achats :", s4)

Les **taux** : étape-à-étape (où est la fuite ?) et global.

In [ ]:
# === À COMPLÉTER ===
t_vue    = round(100.0 * s2 / s1, 1)
t_panier = round(100.0 * ... / ..., 1)
t_achat  = round(100.0 * ... / ..., 1)
t_global = round(100.0 * s4 / s1, 1)

releves["R7_taux_etape_a_etape"] = (t_vue, t_panier, t_achat)
print(f"vue {t_vue} % -> panier {t_panier} % -> achat {t_achat} %")
print(f"taux global : {t_global} %")

### D2 — Segmenter par device puis par ville

Même entonnoir, ventilé. Le mobile (~78 % du trafic) convertit-il mieux ?

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
SELECT device,
       COUNT(DISTINCT session_id)                    AS sessions,
       COUNT(DISTINCT CASE WHEN event_type = 'purchase'
                           THEN session_id END)      AS achats,
       ROUND(100.0 *
         COUNT(DISTINCT CASE WHEN event_type = 'purchase'
                             THEN session_id END)
         / COUNT(DISTINCT session_id), 2)            AS taux_global
FROM   events
GROUP BY device ORDER BY sessions DESC
""").show()

# Meme requete par ville (limitez aux 10 premieres par sessions) :
...

### D3 — Le pivot vers les transactions

`order_id` n'est renseigné que sur les événements `purchase` : c'est le pont
entre comportement et transactions. **Contrôle :** tout `purchase`
pointe-t-il vers une commande connue ?

In [ ]:
# === À COMPLÉTER ===
purchase_orphelins = spark.sql("""
SELECT COUNT(*)
FROM (SELECT * FROM events WHERE event_type = 'purchase') e
LEFT ... JOIN commandes_clean o ON e.order_id = o.order_id
""").first()[0]

releves["R8_purchase_orphelins"] = purchase_orphelins
print("Evenements purchase sans commande :", purchase_orphelins)

**Synthèse de l'entonnoir (à rédiger).** Quatre paragraphes courts :
**constat** (la marche la plus fuyante, chiffres à l'appui), **hypothèse**
(pourquoi ?), **recommandation testable**, **limites** (pensez aux ~30 % de
sessions anonymes et à la nature synthétique des données).

*Votre réponse :* …

## Partie E — Bonus : fonctions fenêtres (15 min)

### E1 — Top 3 produits par région (classement puis filtre)

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventes_par_region_produit AS
SELECT c.region, p.nom_produit,
       SUM(i.quantite * i.prix_unitaire_fcfa) AS ca
FROM   lignes_commande i
JOIN   commandes_clean o ON i.order_id    = o.order_id
JOIN   clients c         ON o.customer_id = c.customer_id
JOIN   produits p        ON i.product_id  = p.product_id
WHERE  o.statut = 'livree'
GROUP BY c.region, p.nom_produit
""")

spark.sql("""
SELECT * FROM (
  SELECT region, nom_produit, ca,
         ROW_NUMBER() OVER (PARTITION BY ...
                            ORDER BY ... DESC) AS rang
  FROM ventes_par_region_produit
) WHERE rang <= 3
ORDER BY region, rang
""").show(30, truncate=False)

### E2 — Évolution mensuelle du CA (LAG)

Le pic de décembre du fil rouge doit apparaître : c'est votre contrôle de
vraisemblance.

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
CREATE OR REPLACE TEMP VIEW ca_mensuel AS
SELECT date_format(date_commande, 'yyyy-MM') AS mois,
       SUM(montant_total_fcfa)               AS ca
FROM   commandes_clean
WHERE  statut = 'livree'
GROUP BY 1
""")

spark.sql("""
SELECT mois, ca,
       ca - LAG(ca, 1) OVER (ORDER BY ...) AS delta
FROM   ca_mensuel ORDER BY mois
""").show(30)

## Relevés finaux et livrable

Ré-affichez le tableau complet : **aucune valeur ne doit rester à `None`**
(R7 exclu si vous n'avez pas atteint le bonus — indiquez-le alors en
commentaire).

In [ ]:
for cle, valeur in releves.items():
    print(f"{cle:35s} : {valeur}")

restants = [k for k, v in releves.items() if v is None]
print("\nReleves manquants :", restants if restants else "aucun - OK")

## Pont vers le livrable

Depuis la racine de votre dépôt :

```bash
git add notebooks/TP5_jointures.ipynb
git commit -m "TP5 : jointures, audit orphelines, entonnoir"
git push
```

**Vérifiez sur github.com** que le notebook s'affiche **avec ses sorties**.

**Avant la séance 6 :** repérez 2–3 jeux de données publics réels
(data.gouv.sn, ANSD, Kaggle, data.humdata.org…) candidats pour votre projet
individuel — le **jalon 0** (fiche de cadrage) sera lancé en séance 6.
Lecture : Reis & Housley, chap. 6 (*Storage*).